In [1]:
import gradio as gr
import requests
import json
import pyttsx3
import speech_recognition as sr
import threading

# OpenRouter API key and model
API_KEY = "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"
MODEL = "deepseek/deepseek-chat-v3-0324:free"

def query_openrouter(messages, temperature=0.7):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": 1024,
        "stream": False
    }
    try:
        response = requests.post("https://openrouter.ai/api/v1/chat/completions",
                                 headers=headers, data=json.dumps(data))
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["message"]["content"]
    except Exception as e:
        return f"⚠️ OpenRouter error: {str(e)}"

def speak_text(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()

def record_and_transcribe():
    recognizer = sr.Recognizer()
    try:
        mic = sr.Microphone()
        with mic as source:
            recognizer.adjust_for_ambient_noise(source)
            print("🎤 Listening...")
            audio = recognizer.listen(source, timeout=5)
        text = recognizer.recognize_google(audio)
        return text
    except Exception as e:
        return f"⚠️ Mic error: {e}"

def alpha_handler(user_input, chat_history, temperature):
    if not user_input:
        return chat_history

    messages = [{"role": "system", "content": "You are a helpful and intelligent assistant."}]
    for user_msg, assistant_msg in chat_history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": user_input})

    reply = query_openrouter(messages, temperature)
    chat_history.append((user_input, reply))
    threading.Thread(target=speak_text, args=(reply,)).start()
    return chat_history

# 🎨 Gradio UI
with gr.Blocks(css=".gradio-container {max-width: 800px; margin: auto;}") as demo:
    gr.Markdown("""
    <div style="text-align: center; padding: 10px;">
        <h1 style="font-size: 2.2em;">🎤🤖 Alpha Voice Assistant</h1>
        <p style="font-size: 1.1em;">Chat with voice or text, powered by DeepSeek</p>
    </div>
    """)

    chatbot = gr.Chatbot(label="💬 Chat History", height=400)
    chat_state = gr.State([])

    with gr.Row():
        manual_input = gr.Textbox(placeholder="Type a message...", show_label=False, container=False, scale=4)
        mic_btn = gr.Button("🎙", scale=1)
        clear_btn = gr.Button("🗑", scale=1)

    temperature = gr.Slider(0, 1, value=0.7, step=0.1, label="🧠 Creativity Level")

    def mic_workflow(chat_history, temperature):
        transcribed = record_and_transcribe()
        return alpha_handler(transcribed, chat_history, temperature)

    mic_btn.click(fn=mic_workflow,
                  inputs=[chat_state, temperature],
                  outputs=chatbot)

    manual_input.submit(fn=alpha_handler,
                        inputs=[manual_input, chat_state, temperature],
                        outputs=chatbot)

    clear_btn.click(lambda: [], None, chatbot)
    clear_btn.click(lambda: [], None, chat_state)

demo.launch()


C:\Users\Rohit BS\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Rohit BS\AppData\Local\Temp\ipykernel_20444\3174104735.py:75: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="💬 Chat History", height=400)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
